# CUDA 12.8 compatibility check

Validates that a **CUDA 12.8** image (the `pytorch` image: `nvcc` 12.8 plus a
CUDA 12.x build of `torch`) works on the GPU node it's scheduled on. Run the
**same notebook, unchanged**, on two nodes and compare:

1. a node on the **r570** driver, whose `nvidia-smi` reports `CUDA Version: 12.8`
2. a node on an **r580+** driver, whose `nvidia-smi` reports `CUDA Version: 13.x`

The image doesn't change between the two runs, only the node's driver. A newer
driver always runs apps built with an older CUDA toolkit (*backward
compatibility*), so **every check should pass on both nodes**. A check that
passes on the first node and fails on the second is a regression introduced by
the driver upgrade.

| Driver on the node | `nvidia-smi` CUDA Version | Compatibility mode for a 12.8 image | Expected |
|---|---|---|---|
| r570 | 12.8 | native (driver matches toolkit) | all PASS |
| r575 | 12.9 | native (driver newer, same major) | all PASS |
| r580, r590, r595, r610, r615 | 13.x | backward compatibility (newer major) | all PASS |
| r525 to r565 | 12.0 to 12.7 | minor version compatibility | PASS, except PTX JIT fails (expected) |
| older than r525 | 11.x | incompatible | FAIL |

Each check prints `PASS`, `FAIL`, `SKIP` or `INFO`. The last cell prints a
summary, writes the results to `cuda-check-results/` so the two runs can be
diffed, and raises if any check failed.

Allocations are kept small (well under 1 GB), so the notebook is safe to run
on a time-sliced or MIG-partitioned GPU.

Run with: **Kernel > Restart Kernel and Run All Cells**, or
`jupyter nbconvert --to notebook --execute cuda_compat_check.ipynb`.

In [ ]:
import ctypes
import glob
import json
import os
import re
import shutil
import socket
import subprocess
import tempfile
from datetime import datetime, timezone

EXPECTED_TOOLKIT = "12.8"
RESULTS = []


def record(name, ok, detail=""):
    status = "PASS" if ok else "FAIL"
    RESULTS.append({"check": name, "status": status, "detail": detail})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail else ""))
    return ok


def skip(name, reason):
    RESULTS.append({"check": name, "status": "SKIP", "detail": reason})
    print(f"  SKIP: {name} -- {reason}")


def info(name, detail):
    RESULTS.append({"check": name, "status": "INFO", "detail": detail})
    print(f"  INFO: {name} -- {detail}")


def run(cmd, **kwargs):
    return subprocess.run(cmd, capture_output=True, text=True, **kwargs)


def cuda_version_str(v):
    """CUDA API version integer -> "major.minor", e.g. 12080 -> "12.8"."""
    return f"{v // 1000}.{(v % 1000) // 10}"


def parse_version(s):
    major, minor = s.split(".")[:2]
    return int(major), int(minor)

## 1. Driver and GPU

What the node provides. The driver's maximum CUDA version comes from
`cuDriverGetVersion()` in `libcuda.so.1`, which the GPU Operator's container
toolkit injects into the pod. This is the same number `nvidia-smi` prints as
"CUDA Version".

In [ ]:
print("== Driver and GPU ==")
info("Hostname", socket.gethostname())
for var in ("CUDA_VISIBLE_DEVICES", "NVIDIA_VISIBLE_DEVICES", "NVIDIA_DRIVER_CAPABILITIES"):
    info(var, os.environ.get(var, "<unset>"))

GPUS = []
if record("nvidia-smi is on PATH", shutil.which("nvidia-smi") is not None):
    out = run(["nvidia-smi",
               "--query-gpu=index,name,driver_version,memory.total,compute_cap,mig.mode.current",
               "--format=csv,noheader"])
    if record("nvidia-smi queried GPU status", out.returncode == 0, out.stderr.strip()):
        for line in out.stdout.strip().splitlines():
            idx, name, driver, memory, cc, mig = [field.strip() for field in line.split(",")]
            GPUS.append({"index": idx, "name": name, "driver": driver,
                         "memory": memory, "compute_cap": cc, "mig": mig})
            info(f"GPU {idx}", f"{name}, {memory}, compute capability {cc}, MIG {mig}")
        record("At least one GPU visible", len(GPUS) > 0)

DRIVER_VERSION = GPUS[0]["driver"] if GPUS else None
COMPUTE_CAP = GPUS[0]["compute_cap"].replace(".", "") if GPUS else None  # e.g. "80"
info("Driver version", DRIVER_VERSION or "<unknown>")

DRIVER_CUDA = None
try:
    libcuda = ctypes.CDLL("libcuda.so.1")
    version = ctypes.c_int()
    if libcuda.cuDriverGetVersion(ctypes.byref(version)) == 0:
        DRIVER_CUDA = cuda_version_str(version.value)
except OSError as e:
    print(f"  could not load libcuda.so.1: {e}")
record("Driver reports its maximum CUDA version", DRIVER_CUDA is not None,
       f"driver {DRIVER_VERSION} supports up to CUDA {DRIVER_CUDA}")

## 2. CUDA toolkit in the image

What the image provides: the `nvcc` compiler, the CUDA runtime library
(`libcudart`) under `/usr/local/cuda`, and the CUDA version `torch` was built
against. These don't change between the two runs.

In [ ]:
print("== CUDA toolkit in the image ==")
NVCC_VERSION = None
if record("nvcc is on PATH", shutil.which("nvcc") is not None):
    out = run(["nvcc", "--version"])
    match = re.search(r"release (\d+\.\d+)", out.stdout)
    NVCC_VERSION = match.group(1) if match else None
    record(f"nvcc reports release {EXPECTED_TOOLKIT}", NVCC_VERSION == EXPECTED_TOOLKIT,
           f"nvcc release {NVCC_VERSION}")

RUNTIME_CUDA = None
cudart_candidates = sorted(glob.glob("/usr/local/cuda/lib64/libcudart.so*"))
if cudart_candidates:
    libcudart = ctypes.CDLL(cudart_candidates[0])
    version = ctypes.c_int()
    if libcudart.cudaRuntimeGetVersion(ctypes.byref(version)) == 0:
        RUNTIME_CUDA = cuda_version_str(version.value)
    record(f"libcudart reports CUDA {EXPECTED_TOOLKIT}", RUNTIME_CUDA == EXPECTED_TOOLKIT,
           f"{cudart_candidates[0]} -> CUDA {RUNTIME_CUDA}")
else:
    record("libcudart found under /usr/local/cuda/lib64", False)

try:
    import torch
    TORCH_CUDA = torch.version.cuda
    info("torch version", torch.__version__)
    record("torch was built with CUDA 12.x", bool(TORCH_CUDA) and TORCH_CUDA.startswith("12."),
           f"torch.version.cuda={TORCH_CUDA}")
except ImportError as e:
    torch = None
    TORCH_CUDA = None
    record("torch is importable", False, str(e))

## 3. Compatibility mode

Compares the driver's maximum CUDA version with each CUDA build in the image
and names which compatibility rule applies:

- **native**: the driver is the same or newer within the same major version.
- **backward compatibility**: the driver is a newer major version, such as a
  CUDA 13 driver running a CUDA 12 app. Fully supported.
- **minor version compatibility**: the driver is older than the app within the
  same major version. Works, but PTX JIT and some newer features don't.
- **incompatible**: the driver is an older major version than the app.

In [ ]:
print("== Compatibility mode ==")


def compat_mode(app_cuda, driver_cuda):
    app, drv = parse_version(app_cuda), parse_version(driver_cuda)
    if drv[0] < app[0]:
        return "incompatible"
    if drv[0] > app[0]:
        return "backward compatibility"
    if drv[1] >= app[1]:
        return "native"
    return "minor version compatibility"


MODES = {}
if DRIVER_CUDA:
    for label, app_cuda in (("CUDA toolkit (nvcc)", NVCC_VERSION), ("torch", TORCH_CUDA)):
        if not app_cuda:
            skip(f"{label} compatibility mode", "version unknown")
            continue
        mode = compat_mode(app_cuda, DRIVER_CUDA)
        MODES[label] = mode
        record(f"{label} CUDA {app_cuda} can run on a CUDA {DRIVER_CUDA} driver",
               mode != "incompatible", mode)
else:
    skip("Compatibility mode", "driver CUDA version unknown")

# PTX JIT needs a driver at least as new as the toolkit that produced the PTX
EXPECT_PTX_JIT = MODES.get("CUDA toolkit (nvcc)") != "minor version compatibility"

## 4. Compile and run a CUDA C++ program with `nvcc`

Builds a vector-add kernel with the image's CUDA 12.8 toolkit two ways:

- **SASS**: compiled ahead of time for this GPU's exact architecture (for
  example `sm_80` on A100, `sm_90` on H200). This is how most libraries ship.
- **PTX only**: intermediate code that the *driver* compiles at load time
  (JIT). This exercises the driver's compiler, and it's the case the CUDA
  compatibility docs call out as failing under minor version compatibility.

The program also prints the driver and runtime versions as CUDA itself sees
them.

In [ ]:
print("== nvcc compile and run ==")
VECTOR_ADD_CU = r"""
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CHECK(call) do { cudaError_t err = (call); if (err != cudaSuccess) { \
    fprintf(stderr, "CUDA error: %s (%s:%d)\n", cudaGetErrorString(err), __FILE__, __LINE__); \
    return 1; } } while (0)

__global__ void vector_add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    int driver = 0, runtime = 0;
    CHECK(cudaDriverGetVersion(&driver));
    CHECK(cudaRuntimeGetVersion(&runtime));
    printf("driver_cuda=%d runtime_cuda=%d\n", driver, runtime);

    const int n = 1 << 20;
    const size_t bytes = n * sizeof(float);
    float *ha = (float *)malloc(bytes), *hb = (float *)malloc(bytes), *hc = (float *)malloc(bytes);
    for (int i = 0; i < n; i++) { ha[i] = (float)i; hb[i] = 2.0f * i; }

    float *da, *db, *dc;
    CHECK(cudaMalloc(&da, bytes));
    CHECK(cudaMalloc(&db, bytes));
    CHECK(cudaMalloc(&dc, bytes));
    CHECK(cudaMemcpy(da, ha, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db, hb, bytes, cudaMemcpyHostToDevice));
    vector_add<<<(n + 255) / 256, 256>>>(da, db, dc, n);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(hc, dc, bytes, cudaMemcpyDeviceToHost));

    int mismatches = 0;
    for (int i = 0; i < n; i++) if (hc[i] != 3.0f * i) mismatches++;
    printf("mismatches=%d\n", mismatches);

    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(ha); free(hb); free(hc);
    return mismatches == 0 ? 0 : 1;
}
"""

if not (NVCC_VERSION and COMPUTE_CAP):
    skip("nvcc compile and run", "needs nvcc and a visible GPU")
else:
    builds = {
        f"SASS (sm_{COMPUTE_CAP})": f"arch=compute_{COMPUTE_CAP},code=sm_{COMPUTE_CAP}",
        f"PTX only (compute_{COMPUTE_CAP}, driver JIT)": f"arch=compute_{COMPUTE_CAP},code=compute_{COMPUTE_CAP}",
    }
    with tempfile.TemporaryDirectory() as tmp:
        src = os.path.join(tmp, "vector_add.cu")
        with open(src, "w") as f:
            f.write(VECTOR_ADD_CU)
        for label, gencode in builds.items():
            binary = os.path.join(tmp, "vector_add_" + gencode.split("=")[-1])
            out = run(["nvcc", "-O2", "-gencode", gencode, src, "-o", binary])
            if not record(f"nvcc builds {label}", out.returncode == 0, out.stderr.strip()[-500:]):
                continue
            out = run([binary])
            output = (out.stdout + out.stderr).strip()
            if label.startswith("PTX") and not EXPECT_PTX_JIT:
                record(f"{label} fails as expected under minor version compatibility",
                       out.returncode != 0, output)
            else:
                record(f"{label} runs and adds vectors correctly", out.returncode == 0, output)

## 5. PyTorch on the GPU

Exercises the CUDA libraries bundled with `torch`. Each GPU result is compared
against the same computation on the CPU:

- **cuBLAS**: matrix multiply
- **cuDNN**: 2D convolution
- **cuFFT**: fast Fourier transform
- **cuRAND**: random number generation
- **end to end**: a short training loop, with backprop and the optimizer step

TF32 is turned off so the float32 GPU results are comparable to the CPU's.

In [ ]:
print("== PyTorch on the GPU ==")
if torch is None:
    skip("PyTorch checks", "torch is not importable")
elif not record("torch.cuda.is_available()", torch.cuda.is_available()):
    skip("PyTorch GPU checks", "torch cannot see a GPU")
else:
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.manual_seed(0)
    cuda = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    info("torch device 0", f"{props.name}, {props.total_memory / 1024**3:.1f} GB, "
                           f"compute capability {props.major}.{props.minor}")
    info("torch arch list", ", ".join(torch.cuda.get_arch_list()))

    try:
        a, b = torch.randn(1024, 1024), torch.randn(1024, 1024)
        gpu = (a.to(cuda) @ b.to(cuda)).cpu()
        record("cuBLAS matmul matches CPU", torch.allclose(gpu, a @ b, rtol=1e-3, atol=1e-3),
               f"max abs diff {(gpu - a @ b).abs().max().item():.2e}")
    except RuntimeError as e:
        record("cuBLAS matmul matches CPU", False, str(e))

    try:
        info("cuDNN version", str(torch.backends.cudnn.version()))
        conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1)
        x = torch.randn(8, 3, 64, 64)
        with torch.no_grad():
            ref = conv(x)
            gpu = conv.to(cuda)(x.to(cuda)).cpu()
        record("cuDNN conv2d matches CPU", torch.allclose(gpu, ref, rtol=1e-3, atol=1e-3),
               f"max abs diff {(gpu - ref).abs().max().item():.2e}")
    except RuntimeError as e:
        record("cuDNN conv2d matches CPU", False, str(e))

    try:
        signal = torch.randn(4096)
        ref = torch.fft.fft(signal)
        gpu = torch.fft.fft(signal.to(cuda)).cpu()
        record("cuFFT fft matches CPU", torch.allclose(gpu, ref, rtol=1e-4, atol=1e-3),
               f"max abs diff {(gpu - ref).abs().max().item():.2e}")
    except RuntimeError as e:
        record("cuFFT fft matches CPU", False, str(e))

    try:
        samples = torch.randn(1_000_000, device=cuda)
        mean, std = samples.mean().item(), samples.std().item()
        record("cuRAND normal samples have mean ~0 and std ~1",
               abs(mean) < 0.01 and abs(std - 1) < 0.01, f"mean {mean:.4f}, std {std:.4f}")
    except RuntimeError as e:
        record("cuRAND normal samples have mean ~0 and std ~1", False, str(e))

    try:
        model = torch.nn.Sequential(
            torch.nn.Linear(256, 512), torch.nn.ReLU(), torch.nn.Linear(512, 1),
        ).to(cuda)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        x = torch.randn(512, 256, device=cuda)
        y = x[:, :8].sum(dim=1, keepdim=True)
        losses = []
        for _ in range(200):
            optimizer.zero_grad()
            loss = torch.nn.functional.mse_loss(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        record("Training loop reduces loss", losses[-1] < 0.5 * losses[0],
               f"loss {losses[0]:.4f} -> {losses[-1]:.4f}")
    except RuntimeError as e:
        record("Training loop reduces loss", False, str(e))

    torch.cuda.empty_cache()

## 6. Summary

Prints every result and saves them to
`cuda-check-results/<hostname>_driver-<version>_<timestamp>.json`. To compare
the runs from the two nodes, diff the two files or load both into a
DataFrame. The cell raises if any check failed.

In [ ]:
counts = {s: sum(r["status"] == s for r in RESULTS) for s in ("PASS", "FAIL", "SKIP", "INFO")}
print("================================")
print(f" Driver {DRIVER_VERSION} (CUDA {DRIVER_CUDA}) | image toolkit {NVCC_VERSION} | torch CUDA {TORCH_CUDA}")
for label, mode in MODES.items():
    print(f" {label}: {mode}")
print(f" {counts['PASS']} passed, {counts['FAIL']} failed, {counts['SKIP']} skipped")
print("================================")
for r in RESULTS:
    if r["status"] in ("FAIL", "SKIP"):
        print(f"  {r['status']}: {r['check']} -- {r['detail']}")

os.makedirs("cuda-check-results", exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
results_path = os.path.join(
    "cuda-check-results", f"{socket.gethostname()}_driver-{DRIVER_VERSION}_{timestamp}.json")
with open(results_path, "w") as f:
    json.dump({
        "hostname": socket.gethostname(),
        "timestamp": timestamp,
        "driver_version": DRIVER_VERSION,
        "driver_cuda": DRIVER_CUDA,
        "nvcc_version": NVCC_VERSION,
        "runtime_cuda": RUNTIME_CUDA,
        "torch_cuda": TORCH_CUDA,
        "gpus": GPUS,
        "compat_modes": MODES,
        "results": RESULTS,
    }, f, indent=2)
print(f"Results written to {results_path}")

if counts["FAIL"]:
    raise AssertionError(f"FAIL: {counts['FAIL']} check(s) failed")
print("PASS: all checks passed")